<a href="https://colab.research.google.com/github/ugisrutinsRSU/RSU_Colab/blob/main/04_Regularization_and_Optimization_Exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Regularisation and Optimisation: House Price Regression

## Overview

In this exercise you will build a **regression neural network** in PyTorch to predict median house prices from census data. Along the way you will apply and compare the regularisation and optimisation techniques introduced in the lecture.

**By the end of this exercise you will have practiced:**
- Building a regression neural network (no sigmoid output — continuous prediction)
- Applying dropout and L2 weight decay to reduce overfitting
- Comparing three optimisers: SGD, Adam, and RMSprop
- Evaluating regression quality with MSE and MAE
- Visualising predictions against ground truth

---

## Dataset: California Housing

The **California Housing dataset** is built into scikit-learn and requires no download. It contains 20,640 census block groups from the 1990 California census.

| Feature | Description |
|---|---|
| `MedInc` | Median income in the block group (in tens of thousands of USD) |
| `HouseAge` | Median house age in the block group |
| `AveRooms` | Average number of rooms per household |
| `AveBedrms` | Average number of bedrooms per household |
| `Population` | Block group population |
| `AveOccup` | Average household occupancy |
| `Latitude` | Block group latitude |
| `Longitude` | Block group longitude |

**Target:** `MedHouseVal` — median house value in hundreds of thousands of USD.

### Key characteristics to keep in mind
- `Population` and `AveOccup` contain extreme outliers (some block groups have very unusual values). This makes feature scaling especially important.
- `Latitude` and `Longitude` encode spatial structure (coastal vs inland, north vs south). A neural network can learn non-linear combinations of these, unlike linear models.
- The target is **capped at 5.0** — the original dataset clips very high-value properties. This means the model will underestimate top-tier prices slightly, which is expected.

## Step 1 — Imports

Import everything you will need upfront. For this exercise you need:
- `numpy`, `pandas`, `matplotlib.pyplot`
- From `sklearn`: `fetch_california_housing`, `train_test_split`, `StandardScaler`
- From `sklearn.metrics`: `mean_squared_error`, `mean_absolute_error`
- `torch`, `torch.nn`, `torch.optim`
- From `torch.utils.data`: `TensorDataset`, `DataLoader`

Also fix random seeds for reproducibility: `torch.manual_seed(42)` and `np.random.seed(42)`.

In [ ]:
# Your imports here


## Step 2 — Load and Explore the Data

Load the dataset using `fetch_california_housing(as_frame=True)` from scikit-learn. This returns a `Bunch` object — access `.frame` to get a single pandas DataFrame with both features and target.

**Tasks:**
1. Load the dataset and display the first few rows.
2. Print the shape and check for missing values.
3. Print summary statistics (`.describe()`). Pay attention to `Population` and `AveOccup` — do their max values look unusual compared to the mean?
4. Plot a histogram of the target column `MedHouseVal`. Notice the spike at 5.0 — this is the cap described above.

In [ ]:
# Load dataset and display first rows


In [ ]:
# Shape, missing values, summary statistics


In [ ]:
# Histogram of MedHouseVal


## Step 3 — Preprocess the Data

### Feature scaling

All eight features are numeric, so no encoding is needed. However, they vary wildly in scale:
- `MedInc` ranges from roughly 0.5 to 15.
- `Population` ranges from 3 to over 35,000.

Without scaling, the gradient updates for parameters connected to `Population` will dominate those connected to smaller-scale features, destabilising training. Use **`StandardScaler`** (zero mean, unit variance).

> **Important:** fit the scaler only on training features, then apply the fitted scaler to the test set. Fitting on the full dataset would leak test-set statistics into training.

**Tasks:**
1. Separate features (`X`) and target (`y`) as NumPy arrays.
2. Split into train (80 %) and test (20 %) sets with `random_state=42`.
3. Fit `StandardScaler` on `X_train` and transform both `X_train` and `X_test`.
4. Convert all four arrays to `torch.FloatTensor`.
5. Wrap into `TensorDataset` objects and create `DataLoader`s with `batch_size=64`, `shuffle=True` for train and `shuffle=False` for test.
6. Print the number of training and test samples to confirm the split.

In [ ]:
# Separate features and target


In [ ]:
# Train/test split, StandardScaler, convert to tensors, create DataLoaders


## Step 4 — Define the Neural Network

### Regression vs classification — the key difference

In the classification exercise, the output layer used a **Sigmoid** activation to squash the output to (0, 1). For regression we want an **unbounded continuous output**, so the output layer has **no activation function** (or equivalently, a linear activation). The model outputs a raw number that we interpret directly as the predicted house price.

### Architecture requirements

```
Input (8 features)
    → Linear(8  → 64)  → ReLU → Dropout(p)
    → Linear(64 → 32)  → ReLU → Dropout(p)
    → Linear(32 → 1)           ← no activation
```

**Tasks:**
1. Define a class `HousePriceNN(nn.Module)` with:
   - `__init__(self, dropout_rate=0.0, l1_lambda=0.0)` — builds the layers and stores `l1_lambda`.
   - `forward(self, x)` — runs the forward pass.
   - `l1_penalty(self)` — computes the L1 weight penalty over weight matrices only (skip biases). Return `torch.tensor(0.0)` when `l1_lambda == 0` so the method is always safe to add to the loss.
2. Instantiate the model with default settings and print it.
3. Count and print the total number of trainable parameters.

> **Tip:** use `model.named_parameters()` in `l1_penalty` and filter for names containing `"weight"` to exclude biases.

In [ ]:
# Define HousePriceNN and instantiate it


## Step 5 — Training Utility

Write a reusable `run_experiment` function so you can run multiple configurations cleanly without copy-pasting the training loop.

**Function signature:**
```python
def run_experiment(label, train_loader, test_loader,
                   dropout_rate=0.0, l1_lambda=0.0, l2_lambda=0.0,
                   optimizer_name="adam", lr=1e-3, epochs=50):
```

**The function should:**
1. Create a fresh `HousePriceNN` with the given `dropout_rate` and `l1_lambda`.
2. Instantiate the chosen optimiser:
   - `"adam"` → `optim.Adam(..., weight_decay=l2_lambda)`
   - `"sgd"` → `optim.SGD(..., momentum=0.9, weight_decay=l2_lambda)`
   - `"rmsprop"` → `optim.RMSprop(..., weight_decay=l2_lambda)`
3. Use `nn.MSELoss()` as the loss criterion.
4. Run a training loop for the given number of epochs:
   - Each batch: `zero_grad → forward → loss + l1_penalty → backward → step`
   - Track average **train MSE** and **test MSE** per epoch.
   - For test loss: use `model.eval()` and `torch.no_grad()`. Evaluate on **raw MSE only** (no penalty) for a fair cross-configuration comparison.
5. After training, compute **test MAE** and **test RMSE**.
6. Print a one-line summary: label, final train MSE, test MSE, test MAE.
7. Return a dict with keys: `label`, `train`, `test`, `mae`, `rmse`.

> **Why MSE for training but report RMSE?** MSE is the loss we optimise (it is differentiable and penalises large errors strongly). RMSE is reported because it is in the same units as the target (hundreds of thousands of USD), making it more interpretable.

In [ ]:
# Define run_experiment


## Step 6 — Experiment A: Baseline vs Regularisation

Run four configurations using **Adam** at `lr=1e-3` for **50 epochs**, varying only the regularisation:

| Config | Dropout | L1 | L2 (weight decay) |
|---|---|---|---|
| Baseline | — | — | — |
| Dropout | 0.3 | — | — |
| L1 | — | 1e-4 | — |
| L2 | — | — | 1e-4 |

After running all four, write a helper function `plot_loss_curves(results, title)` that:
- Creates one subplot per configuration (1 row, 4 columns, shared y-axis).
- Plots train loss (solid) and test loss (dashed) on each subplot.
- Shades the area between the two curves — a larger shaded area signals more overfitting.
- Labels each subplot with the configuration name and its final test RMSE.
- Sets a shared y-axis label (`"MSE Loss"`) and a figure title.

**What to look for:** Which configuration shows the smallest gap between training and test loss? Does any regularisation method hurt performance (test MSE rises)?

In [ ]:
# Run Experiment A


In [ ]:
# Define plot_loss_curves and call it on Experiment A results


## Step 7 — Experiment B: Optimiser Comparison

Now fix regularisation at **Dropout=0.3** (the most general method) and compare three optimisers at two learning rates each:

| Config | Optimiser | LR |
|---|---|---|
| Adam lr=1e-3 | Adam | 0.001 |
| Adam lr=1e-4 | Adam | 0.0001 |
| SGD lr=1e-2 | SGD | 0.01 |
| SGD lr=1e-3 | SGD | 0.001 |
| RMSprop lr=1e-3 | RMSprop | 0.001 |
| RMSprop lr=1e-4 | RMSprop | 0.0001 |

Run all six and plot them.

**What to look for:**
- Does SGD at `lr=1e-2` converge, oscillate, or diverge?
- How does RMSprop compare to Adam at the same learning rate?
- Which optimiser reaches the lowest test MSE fastest (fewest epochs)?

In [ ]:
# Run Experiment B


In [ ]:
# Plot Experiment B results


## Step 8 — Summary Table

Collect all results from both experiments into a single pandas DataFrame with columns:
- `Configuration`
- `Final Train MSE`
- `Final Test MSE`
- `Train-Test Gap` (test MSE − train MSE)
- `Test MAE`
- `Test RMSE`

Print the table. Then print two lines:
- Which configuration achieved the smallest train-test gap.
- Which configuration achieved the lowest test RMSE.

In [ ]:
# Build and print summary table


## Step 9 — Visualise Predictions

Take the best-performing configuration from your summary table and produce a **scatter plot of predicted vs actual house prices** on the test set.

**Tasks:**
1. Retrain (or reuse) the best model.
2. Run inference on the full test set to collect predicted values.
3. Create a scatter plot:
   - x-axis: actual `MedHouseVal`
   - y-axis: predicted `MedHouseVal`
   - Draw a diagonal dashed line representing perfect prediction (`y = x`). Points close to this line indicate good predictions.
   - Colour points by prediction error magnitude (use `c=abs(pred - actual)` and a diverging colormap like `"coolwarm"`).
4. Add axis labels, a title, and a colourbar.

> **What to look for:** Is the model systematically under- or over-predicting? Notice the cluster of points at actual value 5.0 — this is the cap in the data. The model likely underestimates these, which is expected and not a modelling failure.

In [ ]:
# Prediction scatter plot


*Your answers here.*